# Haiku Zero-Shot Evaluation

**Project Name:** Haiku (renamed from Haiku)

## Purpose
- Publication-ready notebook for reproducible training/evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

HAIKU_ROOT = Path('/home/yancui/Haiku')
if str(HAIKU_ROOT / 'src') not in sys.path:

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root='/home/yancui/Haiku')
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
### This is needed to get torch running on the gpu1 queue, if using other GPUs that dont have MIG divisions, dont need to set this var before importing torch
#os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5"
import torch
from torch.utils.data import DataLoader, RandomSampler
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed.nn.functional as dist_fn
from torch import distributed as dist
from torchvision import transforms
from tqdm import tqdm
import wandb
from transformers import BertTokenizer
import pickle
from os.path import join
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
import random
import json

import sys





from models import Haiku, MarkerEmbedding
from data import custom_collate_fn_trimodal, TrimodalDatasetViT, TrimodalDatasetViTEmbedding
from utils import PairwiseCLIPLoss, OneVersusAllLoss, PerChannelSelfStandardization, CustomGaussianBlurTorch
from datetime import timedelta, datetime
import csv

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load yaml config
cfg = OmegaConf.load('/home/yancui/Haiku/src/configs/config.yaml')



In [ ]:
import json
import pandas as pd



#sample_dict = json.load(open('/home/yancui/Haiku/overlap_samples.json'))
sample_dict = json.load(open('/home/yancui/Haiku/src/training/overlap_samples_final.json'))
# sample_ids = sample_ids = pd.read_csv('/project/zhihuanglab/jleiby/codex_clip/sample_ids_with_text.csv', header=None)[0].tolist()
sample_ids = list(sample_dict.keys())


with open('/home/yancui/OmicsAnnotator/test_regions.txt', 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

#sample_ids = test_ids

holdout_list = pd.read_csv('/home/yancui/tier2_acquisition_ids_huanglab (3).csv')['ACQUISITION_ID'].tolist()

overlap_holdout_list = list(set(holdout_list) & set(sample_ids))

new_holdout_list = list(json.load(open('/home/yancui/Haiku/overlap_samples_new.json')).keys())

sample_ids = list(set(test_ids + list(set(overlap_holdout_list) - set(new_holdout_list))))

ref_ids = sorted(sample_ids)

In [ ]:

import os
from tqdm import tqdm

region_metadata_dir = "/data/enable_data/region_metadata"

# First, read all region_metadata CSVs into a dict: {region_id: df}
region_metadata = {}
metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        region_metadata[region_id] = df
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
he_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/he_embedding.pt')
codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/codex_embedding.pt')
region_label = torch.load('/home/yancui/Haiku/res_embdding_126/region_label.pt')
#region_label = torch.load('/home/yancui/Haiku/checkpoints/Trimodal_20250905-2158_full_trainset/region_label.pt')
musk_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/musk_he_embedding.pt')
virtual_codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/virtual_codex_embedding.pt')
#text_embedding = torch.load('/home/yancui/Haiku/res_embdding_124/text_embedding.pt')
#gt_he = torch.load('/home/yancui/Haiku/res_embdding_124/gt_he.pt')

In [ ]:
category_map = {}

category_map['type'] = {
    'normal': 'Normal',
    'nat': 'Normal',
    'at': 'Normal',
    'hyperplasia': 'Benign/Precancerous',
    'malignant': 'Primary Tumor',
    'tumor primary': 'Primary Tumor',
    'metastasis': 'Metastatic Tumor',
    'nan': None,
    '-': None,
    '*': None
}

category_map['grade'] = {
    '1': 'G1',
    '1--2': 'G1',
    'g1': 'G1',
    '2': 'G2',
    '2--3': 'G2',
    'g2': 'G2',
    '3': 'G3',
    'g3': 'G3',
    'nan': None,
    '-': None,
    '*': None
}


In [ ]:
ref_ids = sorted(sample_ids)

metadata_dict_values = {}
metadata_dict_ids = {}

keys = ['tissue_type', 'grade', 'tnm', 'type', 'disease']

for key in keys:
    metadata_dict_values[key] = []
    metadata_dict_ids[key] = []

print(f"Processing {len(region_label)} samples for metadata lookup...")
for i, sample in tqdm(enumerate(region_label), total=len(region_label), desc="Processing metadata"):
    #patch_id = sample['patch_id']
    #print(sample)
    region_id = ref_ids[sample].split('_')[0]
    df = region_metadata.get(region_id, None)
    if df is not None:
        for key in keys:
            if key in df['FEATURE_NAME'].values:
                if (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'nan') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'Unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == np.nan):
                    continue
                else:
                    metadata_dict_values[key].append(df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0])
                    metadata_dict_ids[key].append(i)

In [ ]:
keys = ['tissue_type', 'grade', 'tnm', 'type', 'disease']


def clean_labels(labels, map):
    """
    Map various survival status labels to 'alive' or 'death'.
    """
    mapped = []
    for x in labels:
        if x in map:
            mapped.append(map[x])
        else:
            mapped.append(x)

    return np.array(mapped)


for key in metadata_dict_values:
    filtered_values = []
    filtered_ids = []
    for val, idx in zip(metadata_dict_values[key], metadata_dict_ids[key]):
        if (
            val is not None
            and str(val).lower() != 'nan'
            and str(val).lower() != 'unknown'
            and str(val).lower() != '-'
            and not (isinstance(val, float) and np.isnan(val))
        ):
            filtered_values.append(val)
            filtered_ids.append(idx)
    metadata_dict_values[key] = filtered_values
    metadata_dict_ids[key] = filtered_ids


for key in metadata_dict_values:
    if key in category_map:
        filtered_values = clean_labels(metadata_dict_values[key], category_map[key])
        metadata_dict_ids[key] = np.array(metadata_dict_ids[key])[filtered_values != None]
        metadata_dict_values[key] = np.array(filtered_values)[filtered_values != None]


In [ ]:
np.unique(metadata_dict_values['disease'])


In [ ]:
np.unique(metadata_dict_values['tissue_type'])

In [ ]:
np.unique(metadata_dict_values['type'])

In [ ]:
np.unique(metadata_dict_values['grade'])

In [ ]:
np.unique(metadata_dict_values['tnm'])

In [ ]:
#sample_ids = sample_ids = pd.read_csv('/project/zhihuanglab/jleiby/codex_clip/trimodal_upmc_s4065_train_ids.csv', header=None)[0].tolist()
vocab = pickle.load(open('/data/enable_data/new_individual_samples/final_biomarker_list.pkl', 'rb'))

vocab[vocab == 'PGP9.5'] = 'PGP9_5'

for i in range(len(vocab)):
    if '.' in vocab[i]:
        vocab[i] = vocab[i].replace('.', '_')

cfg.model.vocab = vocab


# Load all .pt embeddings from the directory
esm_embeddings = {}

esm_dir = '/project/zhihuanglab/common/datasets/enable_data/esm_embeddings'

pt_files = [f for f in os.listdir(esm_dir) if f.endswith('.pt')]
for pt_file in pt_files:
    marker = pt_file.replace(".pt", "")
    try:
        embedding_path = os.path.join(esm_dir, pt_file)
        embedding = torch.load(embedding_path)
        esm_embeddings[marker] = embedding
        print(f"Loaded ESM embedding for marker: {marker}")
    except Exception as e:
        print(f"Error loading ESM embedding for {marker}: {str(e)}")

# Determine known markers (present in vocab but not in esm_embeddings)
known_markers = [m for m in vocab if m not in esm_embeddings]

# If no embeddings were loaded, create dummy ones
if not esm_embeddings:
    print("No ESM embeddings found, creating dummy embeddings")
    esm_embeddings = {
        f"marker_{i}": torch.randn(1152) for i in range(10)
    }

# Create marker embedding module
marker_embedding = MarkerEmbedding(
    esm_embeddings,
    known_markers=known_markers,
    embedding_dim=1152,
    model_dim=cfg.model.codex_dim
)


tokenizer = BertTokenizer.from_pretrained(cfg.model.text_model)


# Prep model and dataloader
model = Haiku(
    hf_model=cfg.model.text_model,
    codex_dim=cfg.model.codex_dim,
    text_dim=cfg.model.text_dim,
    he_dim=cfg.model.he_dim,
    projection_dim=cfg.model.projection_dim,
    shared_projection=cfg.model.shared_projection,
    marker_embedding=marker_embedding,
    freeze_bert_layers=True,
    tune_bert_layers=[10, 11],
    freeze_he_encoder=cfg.model.freeze_he_encoder,
    freeze_codex_encoder=cfg.model.freeze_codex_encoder,
    pretrained_weights_path=cfg.model.codex_encoder_weights_path
)


In [ ]:
model.load_state_dict(torch.load('/home/yancui/Haiku/checkpoints/Trimodal_20251203-1815_full_trainset/clip_checkpoint_epoch_49.pth')['model_state_dict'])


In [ ]:
from timm.data.constants import IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD

### Image transforms ###
if cfg.dataset.he_transform:
    he_transform = transforms.Compose([
        transforms.Resize(384, interpolation=3, antialias=True),
        transforms.transforms.CenterCrop((384, 384)),
        transforms.Normalize(
            mean=IMAGENET_INCEPTION_MEAN,
        std=IMAGENET_INCEPTION_STD
        ),
    ])
else:
    he_transform = None

if cfg.dataset.codex_transform:
    codex_transform = [PerChannelSelfStandardization(), CustomGaussianBlurTorch(kernel_size=3, sigma=1.0)]
else:
    codex_transform = None

In [ ]:

sample_data = TrimodalDatasetViTEmbedding(
    cfg.dataset.codex_path,
    cfg.dataset.he_path,
    cfg.dataset.text_path,
    sample_ids,
    tokenizer=tokenizer,
    max_len=cfg.dataset.max_length,
    codex_transform=codex_transform,
    he_transform=he_transform,
    text_captions=cfg.model.text_captions,
    text_caption_sampling=cfg.model.text_caption_sampling,
    #subset_prop=0.1
)


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from tqdm import tqdm
import matplotlib.pyplot as plt

def evaluate_disease_annotation(
    query_embeddings: torch.Tensor,  # (N, D)
    text_embeddings: torch.Tensor,   # (C, D) class text embeddings; row idx = class id
    labels: torch.Tensor,            # (N, 1) or (N,) integer class ids in [0..C-1]
    batch_size: int = 256,
    device: str = "cuda",
    zero_division: float = 0.0       # behavior if a class has no support
):
    """
    Disease annotation via text-image similarity.

    For each sample:
      - cosine similarity to each class text embedding
      - predicted class = argmax(sim)
      - compare to ground-truth label (aligned index)

    Returns mean accuracy, balanced accuracy and F1 (macro & micro).
    """
    assert query_embeddings.ndim == 2, "query_embeddings must be (N, D)"
    assert text_embeddings.ndim == 2, "text_embeddings must be (C, D)"
    N, D = query_embeddings.shape
    C, D2 = text_embeddings.shape
    assert D == D2, "Embedding dims must match"

    # Normalize
    query_embeddings = F.normalize(query_embeddings, dim=1).to(device)   # (N, D)
    text_embeddings = F.normalize(text_embeddings, dim=1).to(device)     # (C, D)

    # Labels to shape (N,)
    if labels.ndim == 2 and labels.shape[1] == 1:
        labels = labels.squeeze(1)
    assert labels.shape == (N,), "labels must be shape (N,) or (N,1)"
    labels = labels.to(device)

    y_true_chunks, y_pred_chunks = [], []

    num_batches = (N + batch_size - 1) // batch_size
    with tqdm(total=num_batches, desc="Evaluating (disease annotation)") as pbar:
        for start in range(0, N, batch_size):
            end = min(start + batch_size, N)
            q = query_embeddings[start:end]           # (B, D)

            # Similarity to all classes: (B, C)
            sims = torch.matmul(q, text_embeddings.T)
            preds = sims.argmax(dim=1)                # (B,)
            gts = labels[start:end]                   # (B,)

            y_true_chunks.append(gts.detach().cpu())
            y_pred_chunks.append(preds.detach().cpu())
            pbar.update(1)

    y_true = torch.cat(y_true_chunks).numpy()
    y_pred = torch.cat(y_pred_chunks).numpy()

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=zero_division)
    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=zero_division)

    results = {
        "accuracy": float(accuracy),
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro),
    }

    print("\n=== Disease Annotation Results ===")
    print(f"Accuracy:   {results['accuracy']:.4f}")
    #print(f"Balanced Accuracy: {results['balanced_accuracy']:.4f}")
    print(f"F1 (macro): {results['f1_macro']:.4f}")
    print(f"F1 (micro): {results['f1_micro']:.4f}")

    return results

def plot_metrics_barplot(methods1, methods2, method_names=None, figsize=(8,5)):
    """
    Draw a barplot to compare different methods' metrics.

    Args:
        methods1: dict, metrics for method 1
        methods2: dict, metrics for method 2
        method_names: list of str, names for each method (optional, default ["Method 1", "Method 2"])
        figsize: tuple, figure size
    """
    import matplotlib.pyplot as plt
    import numpy as np

    metrics = ["accuracy", "balanced_accuracy", "f1_macro", "f1_micro"]

    if method_names is None:
        method_names = ["Method 1", "Method 2"]
    else:
        method_names = list(method_names)

    # Prepare data
    values = []
    for m in [methods1, methods2]:
        values.append([m[metric] for metric in metrics])
    values = np.array(values)  # shape: (2, num_metrics)

    x = np.arange(len(metrics))
    width = 0.35  # bar width

    plt.figure(figsize=figsize)
    for i, (vals, name) in enumerate(zip(values, method_names)):
        plt.bar(x + i*width, vals, width=width, label=name)

    plt.xticks(x + width/2, metrics)
    plt.ylabel("Score")
    plt.ylim(0, 1.05)
    plt.title("Comparison of Methods on Disease Annotation Metrics")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
import numpy as np

def get_random_guess_baseline(y_true):
    """
    Computes the random guess accuracy, balanced accuracy, F1 (macro), and F1 (micro) baseline for a multi-class setting.

    Args:
        y_true (array-like): True labels.

    Returns:
        dict: Dictionary containing 'accuracy', 'balanced_accuracy', 'f1_macro', 'f1_micro' for random guess.
    """
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

    y_true = np.array(y_true)
    n_classes = len(np.unique(y_true))
    n_samples = len(y_true)
    classes = np.unique(y_true)
    y_pred = np.random.choice(classes, size=n_samples)

    acc = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_micro = f1_score(y_true, y_pred, average="micro")

    return {
        "accuracy": acc,
        "balanced_accuracy": balanced_acc,
        "f1_macro": f1_macro,
        "f1_micro": f1_micro
    }

# Example:
# random_baseline = get_random_guess_baseline(disease_label_tensor.numpy())
# print("Random guess baseline:", random_baseline)


In [ ]:
category_map = {}

category_map['type'] = {
    'normal': 'Normal',
    'nat': 'Normal',
    'at': 'Normal',
    'AT': 'Normal',
    "NAT": 'Normal',
    'hyperplasia': 'Benign/Precancerous',
    'malignant': 'Primary Tumor',
    'tumor primary': 'Primary Tumor',
    'Tumor Primary': 'Primary Tumor',
    'Maglignant': 'Primary Tumor',
    'metastasis': 'Metastatic Tumor',
    'nan': None,
    '-': None,
    '*': None
}

category_map['grade'] = {
    '1': 'G1',
    '1--2': 'G1',
    'g1': 'G1',
    '2': 'G2',
    '2--3': 'G2',
    'g2': 'G2',
    '3': 'G3',
    'g3': 'G3',
    'nan': None,
    '-': None,
    '*': None
}


keys = ['tissue_type', 'grade', 'tnm', 'type']


def clean_labels(labels, map):
    """
    Map various survival status labels to 'alive' or 'death'.
    """
    mapped = []
    for x in labels:
        if x in map:
            mapped.append(map[x])
        else:
            mapped.append(x)

    return np.array(mapped)


for key in metadata_dict_values:
    filtered_values = []
    filtered_ids = []
    for val, idx in zip(metadata_dict_values[key], metadata_dict_ids[key]):
        if (
            val is not None
            and str(val).lower() != 'nan'
            and str(val).lower() != 'unknown'
            and str(val).lower() != '-'
            and not (isinstance(val, float) and np.isnan(val))
        ):
            filtered_values.append(val)
            filtered_ids.append(idx)
    metadata_dict_values[key] = filtered_values
    metadata_dict_ids[key] = filtered_ids


for key in metadata_dict_values:
    if key in category_map:
        filtered_values = clean_labels(metadata_dict_values[key], category_map[key])
        metadata_dict_ids[key] = np.array(metadata_dict_ids[key])[filtered_values != None]
        metadata_dict_values[key] = np.array(filtered_values)[filtered_values != None]


In [ ]:
musk_he_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/baseline_musk_he_embedding.pt', weights_only=True)
musk_codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/baseline_musk_codex_embedding.pt', weights_only=True)
musk_text_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/baseline_musk_text_embedding.pt', weights_only=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json
import os

plt.rcParams['svg.fonttype'] = 'none'

keys = ['tissue_type', 'disease']
json_path = "/home/yancui/Haiku/src/training/figs/zero_shot_metrics.json"

# If results file exists, load from json, else compute and save
if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        all_results = json.load(f)
    print(f"Loaded results from {json_path}")
else:
    # Use five different templates for better error bar/statistics estimation.
    template_dict_list = [
        {
            'disease': 'A codex region of {} disease.',
            'tissue_type': 'A codex region of {} tissue type.'
        },
        {
            'disease': '{}',
            'tissue_type': '{}'
        },
        {
            'disease': 'This sample shows {}.',
            'tissue_type': 'This sample is a {} tissue.'
        },
        {
            'disease': 'Histological finding: {}.',
            'tissue_type': 'Histological tissue type: {}.'
        },
        {
            'disease': 'The image presents {}.',
            'tissue_type': 'This image is from {} tissue.'
        }
    ]

    from musk import utils, modeling
    from timm.models import create_model
    from transformers import XLMRobertaTokenizer

    # Normalize embeddings
    codex_embedding = torch.nn.functional.normalize(codex_embedding, p=2, dim=1)
    he_embedding = torch.nn.functional.normalize(he_embedding, p=2, dim=1)
    musk_codex_embedding = torch.nn.functional.normalize(musk_codex_embedding, p=2, dim=1)
    musk_he_embedding = torch.nn.functional.normalize(musk_he_embedding, p=2, dim=1)

    musk = create_model("musk_large_patch16_384")
    utils.load_model_and_may_interpolate("hf_hub:xiangjx/musk", musk, 'model|module', '')

    musk.to('cuda:1')
    musk.eval()

    tokenizer = XLMRobertaTokenizer("/home/yancui/Haiku/src/models/musk_tokenizer.spm")

    from sklearn.preprocessing import LabelEncoder

    all_results = {}

    for key in keys:
        disease_label = metadata_dict_values[key]
        id = metadata_dict_ids[key]
        device = 'cuda'

        lb = LabelEncoder()
        disease_label_tensor = lb.fit_transform(disease_label)
        disease_label_tensor = torch.from_numpy(disease_label_tensor).long()

        text_preprocess = sample_data.text_processing

        codex_metrics_across_templates = []
        he_metrics_across_templates = []
        metrics_list = None  # set in first loop

        for idx, template_dict in enumerate(template_dict_list):
            disease_text_query = []
            template = template_dict[key]
            for disease in lb.classes_:
                disease_text_query.append(template.format(disease))

            text_query = {'text': [], 'att_mask': []}
            for text in disease_text_query:
                input_ids, attention_mask = text_preprocess(text)
                text_query['text'].append(input_ids)
                text_query['att_mask'].append(attention_mask)

            text_query['text'] = torch.stack(text_query['text']).to(device)
            text_query['att_mask'] = torch.stack(text_query['att_mask']).to(device)

            model.to(device)
            model.eval()

            text_embedding = model.get_features_single_modality(text_query, modality='text')
            text_embedding = torch.nn.functional.normalize(text_embedding, p=2, dim=1)

            # Evaluate
            res_codex = evaluate_disease_annotation(codex_embedding[id], text_embedding, disease_label_tensor)
            res_he = evaluate_disease_annotation(he_embedding[id], text_embedding, disease_label_tensor)

            if metrics_list is None:
                metrics_list = list(res_codex.keys())
            codex_metrics_across_templates.append([res_codex[m] for m in metrics_list])
            he_metrics_across_templates.append([res_he[m] for m in metrics_list])

        codex_metrics_across_templates = np.array(codex_metrics_across_templates)  # shape: (n_templates, n_metrics)
        he_metrics_across_templates = np.array(he_metrics_across_templates)

        ours_codex_mean = codex_metrics_across_templates.mean(axis=0)
        ours_codex_std  = codex_metrics_across_templates.std(axis=0)
        ours_he_mean = he_metrics_across_templates.mean(axis=0)
        ours_he_std  = he_metrics_across_templates.std(axis=0)

        # Random guess - (repeat for n_boot to get std)
        n_boot = 10
        rand_all = []
        for _ in range(n_boot):
            rand_metrics = get_random_guess_baseline(disease_label_tensor.numpy())
            rand_all.append([rand_metrics[m] for m in metrics_list])
        rand_all = np.array(rand_all)
        rand_mean = rand_all.mean(axis=0)
        rand_std = rand_all.std(axis=0)

        # Save results to dict for this key
        result = {
            "metrics_names": metrics_list,
            "ours_codex_mean": ours_codex_mean.tolist(),
            "ours_codex_std": ours_codex_std.tolist(),
            "ours_he_mean": ours_he_mean.tolist(),
            "ours_he_std": ours_he_std.tolist(),
            "rand_mean": rand_mean.tolist(),
            "rand_std": rand_std.tolist(),
            "codex_metrics_across_templates": codex_metrics_across_templates.tolist(),
            "he_metrics_across_templates": he_metrics_across_templates.tolist()
        }
        all_results[key] = result

    # Save all results to JSON file
    with open(json_path, 'w') as f:
    print(f"Saved results to {json_path}")



# Always plot from JSON results


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json
import os

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 12

# Only consider these two tasks
keys = ['tissue_type', 'disease']
json_path = "/home/yancui/Haiku/src/training/figs/zero_shot_metrics.json"

with open(json_path, 'r') as f:
    all_results = json.load(f)

# Collect F1 Macro metrics for CODEX for both tasks
f1_macro_codex_means = []
f1_macro_codex_stds = []
rand_f1_macro_means = []
rand_f1_macro_stds = []
plot_labels = []
for key in keys:
    result = all_results[key]
    metrics_list = result["metrics_names"]
    ours_codex_mean = np.array(result["ours_codex_mean"])
    ours_codex_std = np.array(result["ours_codex_std"])
    rand_mean = np.array(result["rand_mean"])
    rand_std = np.array(result["rand_std"])
    # Get index for F1 Macro
    if "F1 Macro" in metrics_list:
        idx = metrics_list.index("F1 Macro")
    elif "F1 macro" in metrics_list:
        idx = metrics_list.index("F1 macro")
    elif "f1_macro" in metrics_list:
        idx = metrics_list.index("f1_macro")
    else:
        raise ValueError("F1 Macro metric not found in metrics_names.")
    f1_macro_codex_means.append(ours_codex_mean[idx])
    f1_macro_codex_stds.append(ours_codex_std[idx])
    rand_f1_macro_means.append(rand_mean[idx])
    rand_f1_macro_stds.append(rand_std[idx])
    if key == "tissue_type":
        plot_labels.append("Tissue Type")
    elif key == "disease":
        plot_labels.append("Disease")
    else:
        plot_labels.append(key)

x = np.arange(len(keys))
width = 0.33

plt.figure(figsize=(8, 4.5))
bars1 = plt.bar(x - width/2, f1_macro_codex_means, width, label='Ours (multi-template)', color='C0', edgecolor='black')
bars2 = plt.bar(x + width/2, rand_f1_macro_means, width, label='Random Guess', color='gray', alpha=0.7, edgecolor='black')
plt.errorbar(x - width/2, f1_macro_codex_means, yerr=f1_macro_codex_stds, fmt='none', ecolor='black', capsize=6, elinewidth=2, capthick=2, zorder=3)
plt.errorbar(x + width/2, rand_f1_macro_means, yerr=rand_f1_macro_stds, fmt='none', ecolor='black', capsize=6, elinewidth=2, capthick=2, zorder=3)
# Print mean values above bars
for i in range(len(x)):
    plt.text(x[i] - width/2, f1_macro_codex_means[i] + f1_macro_codex_stds[i] + 0.01, f"{f1_macro_codex_means[i]:.3f}",
            ha='center', va='bottom', fontweight='bold', fontsize=11)
    plt.text(x[i] + width/2, rand_f1_macro_means[i] + rand_f1_macro_stds[i] + 0.01, f"{rand_f1_macro_means[i]:.3f}",
            ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.ylabel('F1 Macro Score')
plt.title('Zero-Shot F1 Macro (CODEX modality)')
plt.ylim(0, 0.25)
plt.xticks(x, plot_labels, rotation=0)
plt.legend()
# Remove right and upper border
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
plt.show()